In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import glob
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting

# ============================
# Constants and Settings
# ============================
# File pattern: update this regex/pattern for your normal maps.
# This will match filenames such as "circle_0mm_100g_heightmap_linear_detrend_nrm.png"
FILE_PATTERN = "circle_*_nrm.png"

# Histogram settings
BINS = 50

# Colors for plotting histograms
COLOR_ELEVATION = 'skyblue'
COLOR_AZIMUTH = 'salmon'
COLOR_X = 'orchid'
COLOR_Y = 'gold'
COLOR_Z = 'lightgreen'

# Friction model settings
BASE_MU = 0.1      # base friction coefficient
MU_FACTOR = 1.0    # scaling factor for average elevation (in radians)

# Friction cone plotting settings
CONE_RADIUS = 1.0
CONE_RESOLUTION = 50

# ============================
# Class: NormalMap
# ============================
class NormalMap:
    """
    Process a normal map image:
      - Loads the image and converts it from BGR to RGB.
      - Normalizes pixel values to the range [-1,1].
      - Extracts the x, y, z components.
      - Computes spherical coordinates:
          theta: angle between vertical (z) and normal (in radians & degrees)
          phi: azimuth angle (in radians & degrees)
      - Computes an effective friction coefficient from the average elevation.
    """
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.img = None          # Original RGB image
        self.normals = None      # Normalized normal map (values in [-1, 1])
        self.nx = None           # x component
        self.ny = None           # y component
        self.nz = None           # z component
        self.theta = None        # Elevation angle (radians)
        self.phi = None          # Azimuth angle (radians)
        self.theta_deg = None    # Elevation in degrees
        self.phi_deg = None      # Azimuth in degrees

    def load_and_normalize(self) -> None:
        """Load the image file, convert to RGB, and normalize pixel values to [-1,1]."""
        img = cv2.imread(self.file_path, cv2.IMREAD_COLOR)
        if img is None:
            raise IOError(f"Image not found: {self.file_path}")
        # Convert BGR to RGB
        self.img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Normalize: [0,255] -> [0,1] -> [-1,1]
        img_float = self.img.astype(np.float32) / 255.0
        self.normals = img_float * 2.0 - 1.0

    @staticmethod
    def normalize_vectors(nx: np.ndarray, ny: np.ndarray, nz: np.ndarray) -> tuple:
        """Normalize the vector components to ensure unit length."""
        magnitude = np.sqrt(nx**2 + ny**2 + nz**2)
        magnitude[magnitude == 0] = 1.0  # avoid division by zero
        return nx / magnitude, ny / magnitude, nz / magnitude

    @staticmethod
    def compute_spherical(nx: np.ndarray, ny: np.ndarray, nz: np.ndarray) -> tuple:
        """
        Compute spherical coordinates for the normals.
          theta: angle between vertical (z) and the normal [0,pi] (elevation)
          phi: azimuth angle in the x-y plane [-pi, pi]
        """
        theta = np.arccos(nz)
        phi = np.arctan2(ny, nx)
        return theta, phi

    def process(self) -> None:
        """Execute the full processing pipeline for the normal map."""
        self.load_and_normalize()
        # Separate normal components
        self.nx = self.normals[..., 0]
        self.ny = self.normals[..., 1]
        self.nz = self.normals[..., 2]
        # Normalize vectors (in case of numerical issues)
        self.nx, self.ny, self.nz = NormalMap.normalize_vectors(self.nx, self.ny, self.nz)
        # Compute spherical coordinates
        self.theta, self.phi = NormalMap.compute_spherical(self.nx, self.ny, self.nz)
        self.theta_deg = np.degrees(self.theta)
        self.phi_deg = np.degrees(self.phi)

    def compute_effective_mu(self) -> float:
        """
        Compute an effective friction coefficient based on the average elevation angle.
        (Higher average deviation from vertical implies higher friction.)
        """
        avg_theta = np.mean(self.theta)
        return BASE_MU + MU_FACTOR * avg_theta

# ============================
# Class: GraphPlotter
# ============================
class GraphPlotter:
    """
    A collection of static methods to plot:
      - Normal map images and histograms (elevation & azimuth)
      - Component histograms (x, y, z)
      - 3D friction cones for individual surfaces
      - A combined friction cone for two surfaces
    """

    @staticmethod
    def plot_normal_map_histograms(normal_maps: list) -> None:
        """Figure 1: Display each normal map image with its elevation and azimuth histograms."""
        n_images = len(normal_maps)
        fig, axs = plt.subplots(n_images, 3, figsize=(15, 5 * n_images))
        if n_images == 1:
            axs = np.expand_dims(axs, axis=0)
        for idx, nm in enumerate(normal_maps):
            # Display the image
            ax_img = axs[idx, 0]
            ax_img.imshow(nm.img)
            ax_img.set_title(f"Normal Map\n{nm.file_path}")
            ax_img.axis("off")
            # Elevation histogram
            ax_elev = axs[idx, 1]
            ax_elev.hist(nm.theta_deg.flatten(), bins=BINS, density=True,
                         color=COLOR_ELEVATION, edgecolor='black')
            ax_elev.set_xlabel("Elevation Angle (°)")
            ax_elev.set_ylabel("Probability Density")
            ax_elev.set_title("Elevation Distribution")
            # Azimuth histogram
            ax_azim = axs[idx, 2]
            ax_azim.hist(nm.phi_deg.flatten(), bins=BINS, density=True,
                         color=COLOR_AZIMUTH, edgecolor='black')
            ax_azim.set_xlabel("Azimuth Angle (°)")
            ax_azim.set_ylabel("Probability Density")
            ax_azim.set_title("Azimuth Distribution")
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_component_histograms(normal_maps: list) -> None:
        """Figure 2: Plot histograms of the x, y, and z components for each normal map."""
        n_images = len(normal_maps)
        fig, axs = plt.subplots(n_images, 3, figsize=(15, 5 * n_images))
        if n_images == 1:
            axs = np.expand_dims(axs, axis=0)
        for idx, nm in enumerate(normal_maps):
            # X component histogram
            ax_x = axs[idx, 0]
            ax_x.hist(nm.nx.flatten(), bins=BINS, density=True,
                      color=COLOR_X, edgecolor='black')
            ax_x.set_xlabel("X Component")
            ax_x.set_ylabel("Probability Density")
            ax_x.set_title("X Distribution")
            # Y component histogram
            ax_y = axs[idx, 1]
            ax_y.hist(nm.ny.flatten(), bins=BINS, density=True,
                      color=COLOR_Y, edgecolor='black')
            ax_y.set_xlabel("Y Component")
            ax_y.set_ylabel("Probability Density")
            ax_y.set_title("Y Distribution")
            # Z component histogram
            ax_z = axs[idx, 2]
            ax_z.hist(nm.nz.flatten(), bins=BINS, density=True,
                      color=COLOR_Z, edgecolor='black')
            ax_z.set_xlabel("Z Component")
            ax_z.set_ylabel("Probability Density")
            ax_z.set_title("Z Distribution")
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_friction_cone(normal_maps: list, cone_radius: float = CONE_RADIUS,
                             resolution: int = CONE_RESOLUTION) -> None:
        """Figure 3: Plot a 3D friction cone for each normal map individually."""
        n_images = len(normal_maps)
        fig = plt.figure(figsize=(8, 5 * n_images))
        for idx, nm in enumerate(normal_maps):
            mu_eff = nm.compute_effective_mu()
            alpha_eff = np.arctan(mu_eff)
            # Create a mesh in polar coordinates for the cone
            theta_mesh = np.linspace(0, 2 * np.pi, resolution)
            r_mesh = np.linspace(0, cone_radius, resolution)
            theta_grid, r_grid = np.meshgrid(theta_mesh, r_mesh)
            z_grid = r_grid / np.tan(alpha_eff)
            # Convert to Cartesian coordinates
            x_grid = r_grid * np.cos(theta_grid)
            y_grid = r_grid * np.sin(theta_grid)
            ax = fig.add_subplot(n_images, 1, idx + 1, projection='3d')
            ax.plot_surface(x_grid, y_grid, z_grid, alpha=0.5, color='lightblue', edgecolor='none')
            ax.scatter(0, 0, 0, color='red', s=50, label='Apex')
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_zlabel('Z')
            ax.set_title(f'Friction Cone for {nm.file_path}\nEffective μ = {mu_eff:.2f}, '
                         f'α_eff = {np.degrees(alpha_eff):.1f}°')
            ax.legend()
            # Set an equal aspect ratio for clarity
            max_range = np.array([x_grid.max() - x_grid.min(),
                                  y_grid.max() - y_grid.min(),
                                  z_grid.max() - z_grid.min()]).max() / 2.0
            mid_x = (x_grid.max() + x_grid.min()) * 0.5
            mid_y = (y_grid.max() + y_grid.min()) * 0.5
            mid_z = (z_grid.max() + z_grid.min()) * 0.5
            ax.set_xlim(mid_x - max_range, mid_x + max_range)
            ax.set_ylim(mid_y - max_range, mid_y + max_range)
            ax.set_zlim(0, mid_z + max_range)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_combined_friction_cone(nm1: NormalMap, nm2: NormalMap, cone_radius: float = CONE_RADIUS,
                                    resolution: int = CONE_RESOLUTION) -> None:
        """
        New Figure: Combine the effective friction coefficients from two normal maps to create
        a combined friction cone. Here, we add the effective friction coefficients:
            μ_combined = μ_eff(nm1) + μ_eff(nm2)
        and compute the half-angle:
            α_combined = arctan(μ_combined)
        """
        mu_eff1 = nm1.compute_effective_mu()
        mu_eff2 = nm2.compute_effective_mu()
        mu_combined = mu_eff1 + mu_eff2  # Alternatively, an average can be used.
        alpha_combined = np.arctan(mu_combined)
        theta_mesh = np.linspace(0, 2 * np.pi, resolution)
        r_mesh = np.linspace(0, cone_radius, resolution)
        theta_grid, r_grid = np.meshgrid(theta_mesh, r_mesh)
        z_grid = r_grid / np.tan(alpha_combined)
        x_grid = r_grid * np.cos(theta_grid)
        y_grid = r_grid * np.sin(theta_grid)
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(x_grid, y_grid, z_grid, alpha=0.5, color='lightcoral', edgecolor='none')
        ax.scatter(0, 0, 0, color='red', s=50, label='Apex')
        title = (f'Combined Friction Cone\nFrom:\n{nm1.file_path}\n{nm2.file_path}\n'
                 f'μ_combined = {mu_combined:.2f}, α_combined = {np.degrees(alpha_combined):.1f}°')
        ax.set_title(title)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.legend()
        max_range = np.array([x_grid.max() - x_grid.min(),
                              y_grid.max() - y_grid.min(),
                              z_grid.max() - z_grid.min()]).max() / 2.0
        mid_x = (x_grid.max() + x_grid.min()) * 0.5
        mid_y = (y_grid.max() + y_grid.min()) * 0.5
        mid_z = (z_grid.max() + z_grid.min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(0, mid_z + max_range)
        plt.tight_layout()
        plt.show()

# ============================
# Main Script
# ============================
def main():
    # Find all files matching the new file pattern.
    normal_map_files = glob.glob(FILE_PATTERN)
    if len(normal_map_files) < 2:
        raise IOError("Need at least two normal map images to create a combined friction cone.")
    
    # Process each normal map file.
    normal_maps = []
    for file in normal_map_files:
        try:
            nm = NormalMap(file)
            nm.process()
            normal_maps.append(nm)
        except Exception as e:
            print(f"Error processing {file}: {e}")
    
    if not normal_maps:
        print("No valid normal maps to process.")
        return

    # Figure 1: Plot normal map images with elevation and azimuth histograms.
    GraphPlotter.plot_normal_map_histograms(normal_maps)
    
    # Figure 2: Plot x, y, z component histograms.
    GraphPlotter.plot_component_histograms(normal_maps)
    
    # Figure 3: Plot a 3D friction cone for each normal map individually.
    GraphPlotter.plot_friction_cone(normal_maps, cone_radius=CONE_RADIUS, resolution=CONE_RESOLUTION)
    
    # New Figure: Plot a combined friction cone using the first two normal maps.
    GraphPlotter.plot_combined_friction_cone(normal_maps[0], normal_maps[1],
                                               cone_radius=CONE_RADIUS, resolution=CONE_RESOLUTION)

if __name__ == "__main__":
    main()
